In [3]:
#import statements
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import matplotlib.pyplot as plt
from torchinfo import summary
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder
import nilearn.image
import nilearn.plotting
import copy
from torch.utils.data import random_split, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, r2_score
from sklearn.preprocessing import label_binarize
from pathlib import Path
from scipy import signal
import mne
from mne.preprocessing import ICA
from mne_icalabel.iclabel import iclabel_label_components

In [2]:
def parquet_to_edf(parquet_path, output_dir, sfreq=256, units="uV"):
    parquet_path = Path(parquet_path)
    output_dir = Path(output_dir)

    df = pd.read_parquet(parquet_path)

    seg_ids = np.sort(df['segment_index'].unique())
    first_block = df[df['segment_index'] == seg_ids[0]]
    raw_channel_names = first_block['channel_name'].astype(str).tolist()

    # EDF requires unique channel labels; disambiguate duplicates.
    seen = {}
    channel_labels = []
    for ch in raw_channel_names:
        seen[ch] = seen.get(ch, 0) + 1
        channel_labels.append(ch if seen[ch] == 1 else f"{ch} ({seen[ch]})")

    seg_arrays = []
    for seg_id in seg_ids:
        block = df[df['segment_index'] == seg_id]
        rows = list(block.itertuples(index=False))
        if len(rows) != len(raw_channel_names):
            raise ValueError(
                f"segment {seg_id} has {len(rows)} channels, expected "
                f"{len(raw_channel_names)}"
            )
        seg_data = np.stack([np.asarray(r.segment, dtype=np.float64) for r in rows])
        seg_arrays.append(seg_data)

    data = np.concatenate(seg_arrays, axis=1)  # (n_channels, n_samples)

    if units == "uV":
        data = data * 1e-6  # MNE stores EEG in volts

    info = mne.create_info(ch_names=channel_labels, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data, info, verbose=False)

    output_path = output_dir / f"{parquet_path.stem}.edf"
    mne.export.export_raw(
        str(output_path),
        raw,
        fmt='edf',
        overwrite=True,
        verbose=False,
    )
    return output_path

In [16]:
pd.read_parquet("model_data/matched_pca_vectors.parquet")

,subject,vector
0,529993570,"[0.05235492021233305, 0.1443758569858465, -0.1..."
1,419445894,"[-2.267102105757902, -0.4424615769289117, -0.5..."
2,134518856,"[-0.36974932328887516, -0.40325116287329776, 0..."
3,487171266,"[-2.8756415078707027, 1.1385027158972285, -0.7..."
4,170650393,"[-0.5961731375113314, -1.0541028018845164, -0...."
...,...,...
1575,532026665,"[-1.5018083091019372, 1.949326551936844, 0.880..."
1576,968722413,"[-0.5234385108058525, 0.29841154416239996, -0...."
1577,67135355,"[0.9493155163436883, -0.6591801383921884, 1.82..."
1578,892899613,"[0.4545664849225339, -1.2000860736762389, -1.3..."


In [24]:
pd.read_parquet("model_data/orig_eeg_raw/1132274.parquet").head(8)

,subject,n_segments,segment_index,start_index,end_index,original_start_index,original_end_index,sample_rate,original_sample_rate,channel_name,segment
0,1132274,493,176,450560,453119,450560,453119,256.0,256.0,C3,"[-40.6733190234468, -40.14164165059129, -38.01..."
1,1132274,493,176,450560,453119,450560,453119,256.0,256.0,C4,"[-43.33170588772437, -42.800028514868856, -35...."
10,1132274,493,176,450560,453119,450560,453119,256.0,256.0,F3,"[-36.41990004060269, -33.22983580346961, -30.0..."
11,1132274,493,176,450560,453119,450560,453119,256.0,256.0,F4,"[-18.077030677087468, -15.152805126382141, -12..."
20,1132274,493,176,450560,453119,450560,453119,256.0,256.0,O1,"[-37.48325478631372, -41.470835082730076, -45...."
21,1132274,493,176,450560,453119,450560,453119,256.0,256.0,O2,"[-38.8124482184525, -42.5341898284411, -46.787..."
25,1132274,493,176,450560,453119,450560,453119,256.0,256.0,P3,"[-47.58512487056848, -47.053447497712966, -48...."
26,1132274,493,176,450560,453119,450560,453119,256.0,256.0,P4,"[-30.039771566336526, -30.57144893919204, -31...."


In [41]:
pd.read_parquet("model_data/orig_eeg_raw/138699984.parquet")

,subject,n_segments,segment_index,start_index,end_index,original_start_index,original_end_index,sample_rate,original_sample_rate,channel_name,segment
5,138699984,312,138,353280,355837,706554,711668,256,511.9955,C3,"[118.25440229958836, 124.27485439077135, 132.5..."
6,138699984,312,138,353280,355837,706554,711668,256,511.9955,C4,"[118.1829787052524, 121.71304494679605, 129.53..."
9,138699984,312,138,353280,355837,706554,711668,256,511.9955,F3,"[79.7690524579835, 79.90238842211552, 88.52373..."
10,138699984,312,138,353280,355837,706554,711668,256,511.9955,F4,"[95.69923537452293, 98.55957081666847, 105.978..."
17,138699984,312,138,353280,355837,706554,711668,256,511.9955,O1,"[140.26898848798538, 142.67086157728124, 149.6..."
...,...,...,...,...,...,...,...,...,...,...,...
5546,138699984,312,310,795656,798212,1591299,1596412,256,511.9955,F4,"[-25.006943032020914, -22.062722970395384, -15..."
5553,138699984,312,310,795656,798212,1591299,1596412,256,511.9955,O1,"[-27.50402842047917, -22.289323729260467, -31...."
5554,138699984,312,310,795656,798212,1591299,1596412,256,511.9955,O2,"[-27.64601430377067, -21.426875668677837, -24...."
5556,138699984,312,310,795656,798212,1591299,1596412,256,511.9955,P3,"[-16.71834383041362, -11.820888625483716, -15...."


In [4]:
edf_path = parquet_to_edf(
    "model_data/orig_eeg_raw/1132274.parquet",
    output_dir="model_data")

RuntimeError: For exporting to EDF to work, the module edfio is needed, but it could not be imported. Use the following installation method appropriate for your environment:

    pip install edfio
    conda install -c conda-forge edfio